# Técnicas de Prompt Engineering con LLM (GPT)

Este notebook muestra ejemplos prácticos de las técnicas:
- I-O (Input - Output)
- CoT (Chain of Thought)
- CoT-SC (Chain of Thought + Self Consistency)
- ToT (Tree of Thought)

Todos los ejemplos usan la librería `openai` (v1.x) con `client.chat.completions.create()`.

---

## ⚙️ Configuración previa: variables de entorno con `.env`

Para no exponer tu API key directamente en el código, usamos el archivo `.env`.
Funciona igual que en proyectos Node.js.

**Paso 1 — Instala las dependencias** (solo la primera vez):
```
pip install python-dotenv openai
```

**Paso 2 — Crea el archivo `.env`** en la misma carpeta que este notebook:
```
OPENAI_API_KEY=sk-proj-tu-clave-aqui
```

**Paso 3 — Agrega `.env` a tu `.gitignore`** para que nunca se suba a un repositorio:
```
.env
```

In [ ]:
# Instala las dependencias si no las tienes
# Ejecuta esta celda solo una vez
%pip install python-dotenv openai

In [1]:
# Carga las variables del archivo .env y crea el cliente
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()  # Lee el archivo .env de la carpeta actual

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("Cliente configurado correctamente.")

Cliente configurado correctamente.


---

## 1. I-O (Input - Output)

**Concepto:** el prompt contiene únicamente la instrucción o pregunta, sin ejemplos ni razonamiento guiado. Es la forma más directa de interactuar con el modelo y funciona bien para tareas simples y bien definidas como traducción, resumen o clasificación.

In [2]:
prompt = "Traduce esto al francés: 'Hola, ¿cómo estás?'"

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Bonjour, comment ça va ?


---

## 2. CoT (Chain of Thought)

**Concepto:** se le pide al modelo que razone paso a paso antes de dar una respuesta final. La frase clave es *"Pensemos paso a paso"* (*"Let's think step by step"*). Mejora notablemente el desempeño en problemas matemáticos, lógicos o de múltiples pasos.

In [3]:
prompt = (
    "Pregunta: Si hay 3 coches y cada coche tiene 4 ruedas, ¿cuántas ruedas hay en total?\n"
    "Pensemos paso a paso."
)

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Claro, pensemos paso a paso:

1. Hay **3 coches**.  
2. Cada coche tiene **4 ruedas**.  
3. Entonces multiplicamos: **3 × 4 = 12**.

**Respuesta: hay 12 ruedas en total.**


---

## 3. CoT-SC (Chain of Thought + Self-Consistency)

**Concepto:** se generan múltiples respuestas independientes con razonamiento paso a paso y se selecciona la más frecuente (voto mayoritario). Reduce errores ocasionales del modelo aumentando la confiabilidad de la respuesta final.

In [4]:
import collections

prompt = (
    "Pregunta: Una granja tiene 5 corrales y cada uno con 4 cerdos. ¿Cuántos cerdos hay en total?\n"
    "Pensemos paso a paso."
)

respuestas = []

for _ in range(5):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    respuestas.append(response.choices[0].message.content.strip())

conteo = collections.Counter(respuestas)
respuesta_final = conteo.most_common(1)[0][0]

print("Respuesta más frecuente:\n", respuesta_final)

Respuesta más frecuente:
 Claro, pensemos paso a paso:

- Hay **5 corrales**
- En cada corral hay **4 cerdos**

Entonces multiplicamos:

**5 × 4 = 20**

**Respuesta: hay 20 cerdos en total.**


---

## 4. ToT (Tree of Thought)

**Concepto:** el modelo explora múltiples caminos de razonamiento en paralelo, como ramas de un árbol, y evalúa críticamente cada uno. Es útil para problemas abiertos o creativos donde no existe una única solución correcta.

In [5]:
def expand_thought(thought):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": thought}]
    )
    return response.choices[0].message.content

# Ideas iniciales a explorar
ideas = [
    "Mejorar el transporte público",
    "Incentivar el uso compartido de autos",
    "Implementar semáforos inteligentes"
]

tree = {}
for idea in ideas:
    expansion = expand_thought(f"Expande esta idea para reducir el tráfico en una ciudad grande: {idea}")
    evaluacion = expand_thought(f"Evalúa críticamente esta propuesta: {expansion}")
    tree[idea] = {"expansion": expansion, "evaluacion": evaluacion}

for idea, contenido in tree.items():
    print(f"\n{'='*60}")
    print(f"Idea: {idea}")
    print(f"\nExpansión:\n{contenido['expansion']}")
    print(f"\nEvaluación crítica:\n{contenido['evaluacion']}")


Idea: Mejorar el transporte público

Expansión:
Claro. Aquí tienes una versión más desarrollada de la idea **“mejorar el transporte público”** para reducir el tráfico en una ciudad grande:

## Mejorar el transporte público para reducir el tráfico

Una de las formas más efectivas de disminuir la congestión en una gran ciudad es **hacer que el transporte público sea más eficiente, cómodo, rápido y accesible**. Si más personas optan por usar autobuses, trenes, metros o tranvías, habrá menos autos particulares en las calles, lo que ayudará a reducir los embotellamientos.

### ¿Cómo se puede mejorar?
- **Aumentar la frecuencia** de los buses y trenes para que la gente no tenga que esperar tanto.
- **Ampliar rutas y cobertura** para llegar a más barrios, especialmente a zonas alejadas.
- **Crear carriles exclusivos** para el transporte público, permitiendo que se desplace más rápido que los autos.
- **Modernizar vehículos y estaciones** para que sean más cómodos, seguros y limpios.
- **Inte